In [1]:
import os
import sys

# Repo root (trader_intelligence_v2) for .env and paths
_repo_root = os.path.dirname(os.path.abspath(os.curdir)) if os.path.basename(os.path.abspath(os.curdir)) == "Notebooks" else os.path.abspath(os.curdir)

# Load .env from project root so Alpaca keys are available (APCA_API_KEY_ID, APCA_API_SECRET_KEY)
_env_file = os.path.join(_repo_root, ".env")
if os.path.isfile(_env_file):
    try:
        from dotenv import load_dotenv
        load_dotenv(_env_file)
        print(f"Loaded .env from {_env_file}")
    except ImportError:
        print("pip install python-dotenv to load .env")
else:
    print(f"No .env at {_env_file}; using existing env vars")

# Force indicative feed (free) for options; opra requires Algo Trader Plus subscription.
# Remove this line or set APCA_OPTION_FEED=opra in .env to use OPRA if subscribed.
os.environ["APCA_OPTION_FEED"] = "indicative"

# Prefer this repo's backend (same as app, uses alpaca-py) only if alpaca-py is installed in this kernel.
REPO_BACKEND_SRC = os.path.join(_repo_root, "backend", "src")
DEFAULT_ALPACA_PATH = os.path.join(_repo_root, "..", "..", "airflowProd", "notebook", "My_Trader2.0")
ALPACA_NOTEBOOK_PATH = os.path.abspath(os.environ.get("ALPACA_NOTEBOOK_PATH", DEFAULT_ALPACA_PATH))

try:
    import alpaca  # noqa: F401 - alpaca-py must be installed for this repo's backend
    _has_alpaca_py = True
except ModuleNotFoundError:
    _has_alpaca_py = False

if os.path.isdir(ALPACA_NOTEBOOK_PATH):
    sys.path.insert(0, ALPACA_NOTEBOOK_PATH)
if _has_alpaca_py and os.path.isdir(REPO_BACKEND_SRC):
    sys.path.insert(0, REPO_BACKEND_SRC)
    print("Using this repo backend (same as app: alpaca-py for option chain + IV)")
else:
    if not _has_alpaca_py:
        print("alpaca-py not in this kernel; using My_Trader2.0. Install with: pip install alpaca-py")
if os.path.isdir(ALPACA_NOTEBOOK_PATH):
    print(f"My_Trader2.0 on path: {ALPACA_NOTEBOOK_PATH}")

Loaded .env from /home/ken/big_storage/projects/trader_intelligence_v2/.env
alpaca-py not in this kernel; using My_Trader2.0. Install with: pip install alpaca-py
My_Trader2.0 on path: /home/ken/big_storage/airflowProd/notebook/My_Trader2.0


In [2]:
# Reload .env so credentials are set (in case cell 1 wasn't run this session)
import os
_repo = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "Notebooks" else os.getcwd()
_env = os.path.join(_repo, ".env")
if os.path.isfile(_env):
    try:
        from dotenv import load_dotenv
        load_dotenv(_env)
    except ImportError:
        pass

from my_libs_py3.alpaca_trader import AlpacaTrader

alpaca = AlpacaTrader()
print("AlpacaTrader loaded (get_spot_price_from_data_api, calculate_iv_points_from_data_api)")

Unable to load the config, contains a configuration error.
Unable to load the config, contains a configuration error.
Unable to load the config, contains a configuration error.


AlpacaTrader loaded (get_spot_price_from_data_api, calculate_iv_points_from_data_api)


In [3]:
# Confirm paper keys and data URL (run cell 1 first so .env is loaded)
import os
key_id = os.getenv("APCA_API_KEY_ID") or os.getenv("ALPACA_API_KEY")
secret = os.getenv("APCA_API_SECRET_KEY") or os.getenv("ALPACA_API_SECRET")
base = os.getenv("APCA_API_BASE_URL") or os.getenv("ALPACA_API_BASE_URL") or ""
data_base = os.getenv("APCA_DATA_BASE_URL")
option_feed = os.getenv("APCA_OPTION_FEED") or "(default indicative)"
print(f"APCA_API_KEY_ID set: {bool(key_id)} (paper key: {str(key_id or '').startswith('PK')})")
print(f"APCA_API_SECRET_KEY set: {bool(secret)}")
print(f"APCA_API_BASE_URL: {base or '(default paper)'}")
print(f"APCA_DATA_BASE_URL: {data_base or '(inferred from API base)'}")
print(f"APCA_OPTION_FEED: {option_feed}")

APCA_API_KEY_ID set: True (paper key: True)
APCA_API_SECRET_KEY set: True
APCA_API_BASE_URL: https://paper-api.alpaca.markets/
APCA_DATA_BASE_URL: https://data.alpaca.markets
APCA_OPTION_FEED: indicative


## Test run: fetch option chain like the app

The cells below **run** the same code path the app uses: get expirations (if available), then `get_option_chain_df(ticker, expiration_date)` → DataFrame with Strike, Type, Bid, Ask, Last Trade. When using the **repo backend** (alpaca-py), this uses `AlpacaAsyncClient.get_option_chain` under the hood. When using **My_Trader2.0** only, option chain DataFrame is not exposed; run with alpaca-py to test the full flow.

In [11]:
# Test run: same path as the app — get_option_chain_df(ticker, expiration_date)
from datetime import date, timedelta

# Use TICKER from section 1 if already run, else default
try:
    ticker = TICKER
except NameError:
    ticker = "CALM"

# Pick expiration: 1) get_option_expirations if available, 2) My_Trader2.0 raw chain expirations, 3) next Friday
exp_str = None
if hasattr(alpaca, "get_option_expirations") and hasattr(alpaca, "_run_sync"):
    try:
        exps = alpaca._run_sync(alpaca.get_option_expirations(ticker))
        if exps:
            exp_str = exps[0]
    except Exception as e:
        print("get_option_expirations failed:", e)
if not exp_str and hasattr(alpaca, "_get_options_snapshots"):
    # My_Trader2.0: fetch raw snapshots (no filter), extract expirations, use first
    try:
        raw = alpaca._get_options_snapshots(ticker)
        exps = sorted({p.get("expiry") for occ, s in raw.items() if (p := alpaca._parse_occ_symbol(occ)) and p.get("expiry")})
        if exps:
            exp_str = exps[0]
    except Exception:
        pass
if not exp_str:
    d = date.today()
    days_ahead = (4 - d.weekday() + 7) % 7
    if days_ahead == 0:
        days_ahead = 7
    exp_str = (d + timedelta(days=days_ahead)).strftime("%Y%m%d")

# Fetch option chain (sync for My_Trader2.0, _run_sync for repo backend)
if hasattr(alpaca, "_run_sync") and hasattr(alpaca, "get_option_chain_df"):
    chain_df = alpaca._run_sync(alpaca.get_option_chain_df(ticker, exp_str))
else:
    chain_df = alpaca.get_option_chain_df(ticker, exp_str)

# If empty, try known expirations from curl (20260223, 20260225, 20260227)
if chain_df.empty:
    for exp_cand in ["20260223", "20260225", "20260227"]:
        if exp_cand == exp_str:
            continue
        if hasattr(alpaca, "_run_sync"):
            chain_df = alpaca._run_sync(alpaca.get_option_chain_df(ticker, exp_cand))
        else:
            chain_df = alpaca.get_option_chain_df(ticker, exp_cand)
        if not chain_df.empty:
            exp_str = exp_cand
            break

print(f"Option chain for {ticker} exp {exp_str}: {len(chain_df)} rows")
display(chain_df.head(24)) if "display" in dir() else print(chain_df.head(24).to_string())

Option chain for AAPL exp 20260225: 86 rows
     Strike  Type     Bid     Ask  Last Trade        option_symbol
0  285.0000  call  0.0100  0.0500      0.0400  AAPL260225C00285000
1  275.0000  call  0.6600  0.6900      0.7400  AAPL260225C00275000
2  210.0000   put     NaN  0.0300      0.0100  AAPL260225P00210000
3  272.5000   put  2.1600  2.1900      2.0600  AAPL260225P00272500
4  210.0000  call 59.6400 64.2500     54.2300  AAPL260225C00210000
5  235.0000  call 36.3200 39.5200     33.1000  AAPL260225C00235000
6  267.5000   put  0.5600  0.5700      0.5100  AAPL260225P00267500
7  270.0000   put  1.1100  1.1700      1.1400  AAPL260225P00270000
8  242.5000  call 27.9800 31.7400     18.3700  AAPL260225C00242500
9  285.0000   put 11.1600 13.7200     10.9000  AAPL260225P00285000
10 295.0000  call     NaN  0.0100      0.0100  AAPL260225C00295000
11 315.0000  call     NaN  0.0100      0.0100  AAPL260225C00315000
12 282.5000  call  0.0600  0.0700      0.0400  AAPL260225C00282500
13 252.5000   put 

## 1. Spot price (Data API + fallback)

Set `TICKER` below and run.

In [40]:
TICKER = "AAPL"  # Change to any symbol to test

spot = alpaca.get_spot_price_from_data_api(TICKER)
if spot is not None:
    print(f"{TICKER} spot price: {spot}")
else:
    print(f"No spot price for {TICKER}. Check credentials (APCA_API_KEY_ID, APCA_API_SECRET_KEY) and Data API access.")

AAPL spot price: 272.19


## 2. Calculate IV points (7d / 30d / 90d)

Alpaca does not provide IV; we **compute** it from the option chain (Bid/Ask mid or last trade) + spot using the American binomial pricer (pyfin). Same IV points are used by the pipeline that saves to MongoDB for IV rank. **If you see "No IV data"**: option chain may be empty or the stock may have no options (no Finviz fallback).

In [48]:
TICKER="TSLA"
iv_points = alpaca.calculate_iv_points_from_data_api(TICKER)
if iv_points:
    print(f"{TICKER} IV points (calculated from chain):", iv_points)
else:
    print(f"No IV for {TICKER}. Option chain may be empty or stock has no options (no Finviz fallback).")

TSLA IV points (calculated from chain): {'7d': 0.4065, '30d': 0.4171, 'spot': 0.4065}


## 4. Check MongoDB IV data

Query the `iv_history` database (same DB used by the IV pipeline and `calculate_iv_rank` in trader intelligence). Collections are per ticker; documents have `Ticker`, `TimeStamp`, `spot_iv`, and optionally `iv_7d`, `iv_30d`, `iv_90d`.

In [4]:
# Ensure backend is on path for connect_mongo (iv_history)
import os
import sys
from datetime import datetime, timedelta

_repo = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "Notebooks" else os.getcwd()
_backend_src = os.path.join(_repo, "backend", "src")
if _backend_src not in sys.path:
    sys.path.insert(0, _backend_src)

from my_libs_py3 import connect_mongo

# Ticker to check (same as above, or change)
TICKER_IV = os.environ.get("TICKER", "GPRO")

try:
    db = connect_mongo("iv_history", TICKER_IV)
    cutoff = datetime.now() - timedelta(days=365)
    docs = list(db.table.find(
        {"Ticker": TICKER_IV, "TimeStamp": {"$gte": cutoff}},
        {"TimeStamp": 1, "spot_iv": 1, "iv_7d": 1, "iv_30d": 1, "iv_90d": 1}
    ).sort("TimeStamp", 1))

    print(f"MongoDB iv_history / {TICKER_IV}: {len(docs)} documents (last 365 days)")
    if docs:
        with_spot = [d for d in docs if d.get("spot_iv") is not None]
        print(f"  With spot_iv: {len(with_spot)}")
        if with_spot:
            ivs = [d["spot_iv"] for d in with_spot]
            print(f"  spot_iv range: {min(ivs):.4f} – {max(ivs):.4f}")
            print(f"  Latest: {with_spot[-1].get('TimeStamp')}  spot_iv={with_spot[-1].get('spot_iv'):.4f}")
        print("  Sample (last 3):")
        for d in docs[-3:]:
            print(f"    {d.get('TimeStamp')}  spot_iv={d.get('spot_iv')}  iv_7d={d.get('iv_7d')}  iv_30d={d.get('iv_30d')}  iv_90d={d.get('iv_90d')}")
    else:
        print("  No IV history yet. Run the IV pipeline (e.g. collect_iv_history in My_Trader2.0) to populate.")
except Exception as e:
    print(f"MongoDB iv_history check failed: {e}")
    print("  (Ensure MongoDB is reachable, e.g. mongodb://10.0.1.22:27017)")

MongoDB iv_history / GPRO: 2 documents (last 365 days)
  With spot_iv: 2
  spot_iv range: 1.5197 – 2.1609
  Latest: 2026-02-26 00:00:00+00:00  spot_iv=2.1609
  Sample (last 3):
    2026-02-25 00:00:00+00:00  spot_iv=1.5197  iv_7d=1.5197  iv_30d=1.046  iv_90d=1.2705
    2026-02-26 00:00:00+00:00  spot_iv=2.1609  iv_7d=2.1609  iv_30d=1.0714  iv_90d=1.3177


## 5. IV Rank (same as trader intelligence)

Uses the **exact same logic** as `calculate_iv_rank(ticker, lookback_days=252)` in `backend/src/analytics/analysis.py`: query MongoDB `iv_history`, require ≥30 days of `spot_iv`, then rank = (current − min) / (max − min) in [0, 1]. Inlined here so the notebook does not need alpaca-py (importing the full analysis module would pull in alpaca).

In [72]:
# Exact same logic as backend/src/analytics/analysis.py calculate_iv_rank()
# (no import of analysis.py so we avoid alpaca / my_libs_py3 __init__)
import os
import sys
from datetime import datetime, timedelta

_repo = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "Notebooks" else os.getcwd()
_backend_src = os.path.join(_repo, "backend", "src")
if _backend_src not in sys.path:
    sys.path.insert(0, _backend_src)

from my_libs_py3 import connect_mongo

# Ticker (use TICKER_IV from section 4 if you ran it, or set here)
ticker = "TSLA"
lookback_days = 252
docs = []
try:
    db = connect_mongo("iv_history", ticker)
    cutoff = datetime.now() - timedelta(days=lookback_days + 30)
    docs = list(db.table.find(
        {"Ticker": ticker, "TimeStamp": {"$gte": cutoff}},
        {}
        # {"TimeStamp": 1, "spot_iv": 1, 'iv_7d': 1, 'iv_30d': 1, 'iv_90d': 1}
    ).sort("TimeStamp", 1))
    iv_values = [d["spot_iv"] for d in docs if d.get("spot_iv") is not None]
    if len(iv_values) < 30:
        iv_rank = None
    else:
        current_iv = iv_values[-1]
        min_iv, max_iv = min(iv_values), max(iv_values)
        iv_rank = 0.5 if max_iv == min_iv else (current_iv - min_iv) / (max_iv - min_iv)
except Exception:
    iv_rank = None

docs

[{'_id': ObjectId('699e490c2394e7b89bea1581'),
  'TimeStamp': datetime.datetime(2026, 2, 25, 0, 0, tzinfo=<bson.tz_util.FixedOffset object at 0x7fdea7a21cd0>),
  'Ticker': 'TSLA',
  'iv_30d': 0.4264,
  'iv_7d': 0.4161,
  'iv_90d': None,
  'spot_iv': 0.4161}]

### 2c. If you see "No IV" — diagnose

Run this to see exactly what Alpaca returns: URL, feed, snapshot count, and whether each snapshot has quote/IV. With an options subscription we use **feed=opra** (set `APCA_OPTION_FEED=indicative` to force indicative).

In [14]:
if hasattr(alpaca, "diagnose_iv_fetch"):
    diag = alpaca.diagnose_iv_fetch(TICKER)
    for k, v in diag.items():
        print(f"  {k}: {v}")
else:
    print("Using app backend (alpaca-py); no diagnose_iv_fetch. Run section 2 to test IV.")

  url: https://data.sandbox.alpaca.markets/v1beta1/options/snapshots/AAPL
  params: {'feed': 'opra', 'limit': 1000, 'expiration_date_gte': '2026-03-21', 'expiration_date_lte': '2026-03-29'}
  data_base_url: https://data.sandbox.alpaca.markets
  spot: 266.69
  http_status: 401
  snapshot_count: 0
  sample_occ: None
  sample_keys: None
  sample_has_quote: False
  sample_has_iv: False
  sample_mid: None
  response_preview: <html>
<head><title>401 Authorization Required</title></head>
<body>
<center><h1>401 Authorization Required</h1></center>
<hr><center>nginx</center>
</body>
</html>

  auth_note: 401 = credentials rejected. data.sandbox needs PAPER keys; data.alpaca.markets needs LIVE keys. Match APCA_API_BASE_URL (paper vs live) to your key type, or set APCA_DATA_BASE_URL.


## 2b. IV via pipeline logic (Alpaca only; no Finviz fallback)

Pipeline: **calculate** IV from Alpaca option chain only. If the stock has no options or the chain is empty, no IV is returned. Same structure is saved to MongoDB for IV rank when available.

In [67]:
# Pipeline logic from My_Trader2.0/dataCollector.py (_fetch_iv_points). Alpaca only; no Finviz fallback.
def _fetch_iv_points(ticker, alpaca):
    """Calculate IV from Alpaca chain only. Returns dict with spot/7d/30d/90d or None."""
    iv_points = alpaca.calculate_iv_points_from_data_api(ticker)
    if iv_points:
        return iv_points
    return None

iv_points = _fetch_iv_points(TICKER, alpaca)
if iv_points:
    print(f"{TICKER} IV points (Alpaca):", iv_points)
else:
    print(f"No IV for {TICKER}. Stock may have no options (pipeline skips such tickers).")

TSLA IV points (Alpaca): {'7d': 0.4303, '30d': 0.4199, 'spot': 0.4303}


## 3. Quick batch (optional)

Test a few tickers to confirm pipeline behavior.

In [ ]:
TEST_TICKERS = ["AAPL", "SPY", "MSFT"]

for t in TEST_TICKERS:
    spot = alpaca.get_spot_price_from_data_api(t)
    iv = alpaca.calculate_iv_points_from_data_api(t)
    print(f"{t}: spot={spot}, iv_ok={iv is not None}")